In [1]:
import openeo
import json

import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal
import numpy as np

import openeo

bbox = {
        "west":  18.51635,
        "south": 48.78376,
        "east":  18.80255,
        "north": 49.04104
        }

In [2]:
connection = openeo.connect(url="openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [17]:
print(connection.list_collections())

[{'description': 'Copernicus Global 30 meter Digital Elevation Model dataset.', 'extent': {'spatial': {'bbox': [[-180, -90, 180, 90]]}, 'temporal': {'interval': [['2010-12-12T00:00:00Z', '2015-01-16T00:00:00Z']]}}, 'id': 'COPERNICUS_30', 'keywords': ['Copernicus', 'ESA', 'Satellite', 'Global', 'DEM', 'EU', 'EC'], 'license': 'proprietary', 'links': [{'href': 'https://stac.dataspace.copernicus.eu/v1/collections/cop-dem-glo-30-dged-cog/items', 'rel': 'items', 'type': 'application/geo+json'}, {'href': 'https://stac.dataspace.copernicus.eu/v1/', 'rel': 'parent', 'type': 'application/json'}, {'href': 'https://stac.dataspace.copernicus.eu/v1/', 'rel': 'root', 'type': 'application/json'}, {'href': 'https://stac.dataspace.copernicus.eu/v1/collections/cop-dem-glo-30-dged-cog', 'rel': 'self', 'type': 'application/json'}, {'href': 'https://dataspace.copernicus.eu/sites/default/files/media/files/2025-06/copernicus_contributing_mission_data_access_v2_cop_dem_licenses.pdf', 'rel': 'license', 'title':

In [3]:
s2cube = connection.load_collection(
    "SENTINEL2_L2A",
    temporal_extent=["2020-06-01", "2020-11-01"],
    spatial_extent=bbox,
    bands=["B04", "B08", "SCL"],
)

red = s2cube.band("B04")
nir = s2cube.band("B08")
ndvi = (nir - red) / (nir + red)

In [4]:
scl = s2cube.band("SCL")
mask = (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10) | (scl == 11)

In [11]:
# 2D gaussian kernel
g = scipy.signal.windows.gaussian(11, std=1.6)
kernel = np.outer(g, g)
kernel = kernel / kernel.sum()

# Morphological dilation of mask: convolution + threshold
mask = mask.apply_kernel(kernel)
mask = mask > 0.1

ndvi_masked = ndvi.mask(mask)

In [12]:
udf = openeo.UDF(
    """
from scipy.signal import savgol_filter
from openeo.udf import XarrayDataCube

def apply_datacube(cube: XarrayDataCube, context: dict) -> XarrayDataCube:
    array = cube.get_array()
    filled = array.interpolate_na(dim='t')
    smoothed_array = savgol_filter(filled.values, 5, 2, axis=0)
    return DataCube(xarray.DataArray(smoothed_array, dims=array. dims,coords=array.coords))
"""
)


In [13]:
ndvi_smoothed = ndvi_masked.apply_dimension(code=udf, dimension="t")
time_serie_ndvi = ndvi_smoothed.aggregate_temporal_period(
    period="month",
    reducer="median"
)

In [16]:

job = time_serie_ndvi.create_job(
    out_format="GTiff",
    title="NDVI 2020 monthly",
    description="NDVI monthly composites for 2020 (Jun-Oct)"
)
job.start_and_wait()


0:00:00 Job 'j-2604231406034b9fbb3ae4e78506fe29': send 'start'
0:00:19 Job 'j-2604231406034b9fbb3ae4e78506fe29': created (progress 0%)
0:00:24 Job 'j-2604231406034b9fbb3ae4e78506fe29': created (progress 0%)
0:00:30 Job 'j-2604231406034b9fbb3ae4e78506fe29': created (progress 0%)
0:00:39 Job 'j-2604231406034b9fbb3ae4e78506fe29': created (progress 0%)
0:00:49 Job 'j-2604231406034b9fbb3ae4e78506fe29': created (progress 0%)
0:01:01 Job 'j-2604231406034b9fbb3ae4e78506fe29': created (progress 0%)
0:01:17 Job 'j-2604231406034b9fbb3ae4e78506fe29': created (progress 0%)
0:01:36 Job 'j-2604231406034b9fbb3ae4e78506fe29': running (progress N/A)
0:02:00 Job 'j-2604231406034b9fbb3ae4e78506fe29': running (progress N/A)
0:02:30 Job 'j-2604231406034b9fbb3ae4e78506fe29': running (progress N/A)
0:03:07 Job 'j-2604231406034b9fbb3ae4e78506fe29': running (progress N/A)
0:03:55 Job 'j-2604231406034b9fbb3ae4e78506fe29': running (progress N/A)
0:04:53 Job 'j-2604231406034b9fbb3ae4e78506fe29': running (progress 

JobFailedException: Batch job 'j-2604231406034b9fbb3ae4e78506fe29' didn't finish successfully. Status: error (after 0:09:00).

In [15]:
results = job.get_results()
results.download_files("~/ndvi_cleadned_monthly_2020/")

[PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-06-01Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-06-03Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-06-06Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-06-08Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-06-11Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-06-13Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-06-16Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-06-18Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-06-21Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-06-23Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-06-26Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-06-28Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-07-01Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-07-03Z.tif'),
 PosixPath('~/ndvi_cleadned_monthly_2020/openEO_2020-07-06Z.ti